In [1]:
import os
from langchain_text_splitters import RecursiveCharacterTextSplitter
import fitz  # PyMuPDF

e:\india_project\rag-aws\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
### 
# pdf_path = "../instructions/AWS_Customer_Agreement.pdf"
pdf_path = "E:/india_project/rag-aws/instructions/AWS_Customer_Agreement.pdf"
data_path = "E:/india_project/rag-aws/data"

In [4]:
### 1. document loader

def parse_and_chunk(pdf_path: str) -> list[dict]:
    """Parse PDF and return list of chunk dicts with text + metadata."""
    doc = fitz.open(pdf_path)
    full_text_pages = []
    for page_num, page in enumerate(doc, start=1):
        text = page.get_text("text").strip()
        if text:
            full_text_pages.append({"text": text, "page": page_num})

    splitter = RecursiveCharacterTextSplitter(
        chunk_size=500,
        chunk_overlap=50,
        separators=["\n\n", "\n", ". ", " "],
    )

    chunks = []
    for page_data in full_text_pages:
        splits = splitter.split_text(page_data["text"])
        for split in splits:
            chunks.append({
                "text": split,
                "page": page_data["page"],
                "chunk_index": len(chunks),
            })
    return chunks

In [5]:
chunks = parse_and_chunk(pdf_path)

In [6]:
# retriever.py
import os
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings

MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
CHROMA_DIR = f"{data_path}/chroma_db"
COLLECTION_NAME = "docs_chunks"


class Retriever:
    def __init__(self):
        self.model = SentenceTransformer(MODEL_NAME)
        self.client = chromadb.PersistentClient(
            path=CHROMA_DIR,
            settings=Settings(anonymized_telemetry=False),
        )
        self.collection = self.client.get_or_create_collection(
            name=COLLECTION_NAME,
            metadata={"hnsw:space": "cosine"},  # cosine similarity
        )

    def build_index(self, chunks: list[dict]):
        """Embed all chunks and upsert into ChromaDB. Persists automatically."""
        # Clear existing data so re-ingestion is idempotent
        self.client.delete_collection(COLLECTION_NAME)
        # self.collection = self.client.get_or_create_collection(
        #     name=COLLECTION_NAME,
        #     metadata={"hnsw:space": "cosine"},
        # )

        texts = [c["text"] for c in chunks]
        embeddings = self.model.encode(
            texts, show_progress_bar=True, batch_size=32
        ).tolist()

        self.collection.upsert(
            ids=[str(i) for i in range(len(chunks))],
            embeddings=embeddings,
            documents=texts,
            metadatas=[{k: v for k, v in c.items() if k != "text"} for c in chunks],
        )

    def load_index(self):
        """Verify the persisted collection exists and has data."""
        count = self.collection.count()
        if count == 0:
            raise FileNotFoundError("No index found. Call POST /ingest first.")

    def retrieve(self, query: str, top_k: int = 4) -> list[dict]:
        """Return top-k most relevant chunks for a query."""
        q_emb = self.model.encode([query]).tolist()
        results = self.collection.query(
            query_embeddings=q_emb,
            n_results=top_k,
            include=["documents", "metadatas", "distances"],
        )

        chunks = []
        for doc, meta, dist in zip(
            results["documents"][0],
            results["metadatas"][0],
            results["distances"][0],
        ):
            chunk = {"text": doc, **meta}
            chunk["score"] = 1 - dist  # cosine distance → similarity
            chunks.append(chunk)

        return chunks

In [7]:
# generator.py

from openai import OpenAI
import os 
from dotenv import load_dotenv

load_dotenv()


def model_client():
    """Return an OpenAI client for the generator model."""
    
    client = OpenAI(
            base_url=os.getenv(BASE_URL),
            api_key=os.getenv(OPENROUTER_API_KEY),
        )
        # Wrap for LangSmith tracing
    return client



SYSTEM_PROMPT = """You are a precise document assistant. Answer questions ONLY using the
provided context excerpts. If the answer cannot be found in the context, respond with exactly:
"I cannot find the answer to this question in the provided document."
Do not make up facts. Do not use outside knowledge."""

def build_prompt(query: str, chunks: list[dict]) -> str:
    """Construct a grounded RAG prompt from retrieved chunks."""
    context_parts = []
    for i, chunk in enumerate(chunks, start=1):
        context_parts.append(f"[Source {i} — Page {chunk['page']}]\n{chunk['text']}")
    context = "\n\n".join(context_parts)
    return f"{SYSTEM_PROMPT}\n\nContext:\n{context}\n\nQuestion: {query}\n\nAnswer:"

def answer_not_found(response: str) -> bool:
    """Detect if the LLM indicated it could not find an answer."""
    markers = [
        "cannot find", "not found in the document", "not mentioned",
        "no information", "I don't know", "not in the provided",
    ]
    return any(m.lower() in response.lower() for m in markers)


def generate_answer(query: str, chunks: list[dict]) -> str:
    """Run the full generation step and return the answer string."""
    client = model_client()
    response = client.chat.completions.create(
        model=os.getenv(MODEL_NAME),
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user",   "content": build_prompt(query, chunks)},
        ],
        temperature=0.1,
        max_tokens=512,
    )
    return response.choices[0].message.content.strip()

In [8]:
# inside /ask endpoint

retriever = Retriever()
chunks = parse_and_chunk(pdf_path)

query = "what is the aws customer agreement"

chunks = retriever.retrieve(query, top_k=4)
answer = generate_answer(query, chunks)
found  = not answer_not_found(answer)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4880.95it/s]


NameError: name 'BASE_URL' is not defined